# DyslexAI Notebook Handoff

This notebook is preserved as the original research and experimentation source for the project.

The production-quality implementation now lives in the structured codebase:
- `dyslexia-backend/` contains the OCR, correction, API, database, and pipeline modules.
- `frontend/` contains the polished dashboard and workspace UI.
- `docs/notebook_audit.md` explains what was reused from this notebook.

Use this notebook for research notes, demonstrations, and quick experiments. Use dyslexia-backend and frontend folders to run the real local-first application.

In [ ]:
# Example: production pipeline usage from dyslexia-backend
# Run the FastAPI app from the terminal for the full product experience.

from pathlib import Path

# This snippet is intentionally lightweight. It documents how the notebook now
# connects to the extracted code instead of duplicating production logic here.
example_image = Path("dyslexia-backend/data/ocr/uploads/example.png")
        print("Production code now lives in dyslexia-backend/app and frontend/src")
print(f"Example input path: {example_image}")

In [1]:
!pip install python-doctr



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cleaned and Consolidated OCR Pipeline

This cell contains the fully refactored, deduplicated, and functional code for the DyslexAI pipeline.

In [1]:
import os
import re
import cv2
import torch
import matplotlib.pyplot as plt
from statistics import median
from doctr.models import ocr_predictor
from transformers import TrOCRProcessor, VisionEncoderDecoderModel, AutoTokenizer, AutoModelForSeq2SeqLM
from groq import Groq
import zipfile
import shutil

# --- Configuration ---
# Load from environment (set GROQ_API_KEY in .env or system env)
try:
    from dotenv import load_dotenv
    load_dotenv()
    load_dotenv("dyslexia-backend/.env")
except ImportError:
    pass
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "your_api_key_here")
LOCAL_ZIPPED_MODEL_PATH = "DyslexAI_Best_Model (1).zip"
EXTRACTION_PATH = "./DyslexAI_Best_Best_Model_unzipped"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- 1. Load OCR Models (DocTR + TrOCR) ---
def load_ocr_models():
    print("Loading DocTR line detector...")
    detection_model = ocr_predictor(pretrained=True).to(device).eval()

    print("Loading BASE TrOCR Large Handwritten...")
    base_model_name = "microsoft/trocr-large-handwritten"
    base_processor = TrOCRProcessor.from_pretrained(base_model_name)
    base_model = VisionEncoderDecoderModel.from_pretrained(base_model_name).to(device).eval()

    print("[OK] OCR Models loaded successfully")
    return detection_model, base_processor, base_model

# --- 2. Load Local Dyslexia T5 Model ---
def load_local_t5_model():
    tokenizer = None
    model = None

    if os.path.exists(LOCAL_ZIPPED_MODEL_PATH):
        print(f"Attempting to unzip model from {LOCAL_ZIPPED_MODEL_PATH} to {EXTRACTION_PATH}...")
        try:
            os.makedirs(EXTRACTION_PATH, exist_ok=True)
            with zipfile.ZipFile(LOCAL_ZIPPED_MODEL_PATH, 'r') as zip_ref:
                zip_ref.extractall(EXTRACTION_PATH)
            print("[OK] Local T5 Model unzipped successfully.")

            print("Loading local T5 model...")
            tokenizer = AutoTokenizer.from_pretrained(EXTRACTION_PATH)
            model = AutoModelForSeq2SeqLM.from_pretrained(EXTRACTION_PATH).to(device)
            print("[OK] Local T5 Model loaded.")
        except Exception as e:
            print(f"[ERROR] Error during unzipping or loading local T5 model: {e}")
    else:
        print(f"[WARN] Local zipped model '{LOCAL_ZIPPED_MODEL_PATH}' not found. Skipping local T5 step.")
    
    return tokenizer, model

# --- 3. Initialize Groq Client ---
def initialize_groq_client():
    if not GROQ_API_KEY or GROQ_API_KEY == "your_api_key_here":
        print("[WARN] Groq API key not set or invalid. Skipping Groq layer.")
        return None
    try:
        client = Groq(api_key=GROQ_API_KEY)
        print("[OK] Groq client initialized.")
        return client
    except Exception as e:
        print(f"[ERROR] Failed to initialize Groq client: {e}")
        return None


# --- Inference Pipeline Components ---
def run_ocr_inference(image_path, detection_model, processor, trocr_model, model_name="TrOCR"):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image: {image_path}")
        
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_gray_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    height, width = img_rgb.shape[:2]

    result = detection_model([img_gray_rgb])
    results = []
    line_count = 0
    boxed_img = img_rgb.copy()

    for page in result.pages:
        for block in page.blocks:
            for line in block.lines:
                (x0, y0), (x1, y1) = line.geometry
                x_min, y_min = int(x0 * width), int(y0 * height)
                x_max, y_max = int(x1 * width), int(y1 * height)

                pad = 5
                x_min = max(0, x_min - pad)
                y_min = max(0, y_min - pad)
                x_max = min(width, x_max + pad)
                y_max = min(height, y_max + pad)

                crop = img_gray_rgb[y_min:y_max, x_min:x_max]
                if crop.shape[0] < 5 or crop.shape[1] < 5:
                    continue

                line_count += 1
                pixel_values = processor(images=crop, return_tensors="pt").pixel_values.to(device)
                with torch.no_grad():
                    generated_ids = trocr_model.generate(pixel_values)
                text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

                y_center = (y_min + y_max) / 2.0
                results.append({
                    "line_number": line_count,
                    "bbox": (x_min, y_min, x_max, y_max),
                    "y_center": y_center,
                    "text": text
                })

                cv2.rectangle(boxed_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                # print(f"[{model_name} - Box {line_count}] {text}")

    # Sort lines
    results_sorted = sorted(results, key=lambda r: (r["y_center"], r["bbox"][0]))
    heights = [r["bbox"][3]-r["bbox"][1] for r in results_sorted]
    if heights:
        median_h = median(heights)
        clustered = []
        current_cluster = [results_sorted[0]]
        for r in results_sorted[1:]:
            if abs(r["y_center"] - current_cluster[-1]["y_center"]) <= (median_h * 0.5):
                current_cluster.append(r)
            else:
                clustered.extend(sorted(current_cluster, key=lambda x: x["bbox"][0]))
                current_cluster = [r]
        clustered.extend(sorted(current_cluster, key=lambda x: x["bbox"][0]))
        results_sorted = clustered

    paragraph = " ".join([r["text"].strip() for r in results_sorted if r["text"].strip()]).strip()
    return paragraph, boxed_img

def layer_1_sanitize(text):
    if not text: return ""
    text = text.replace(" . ", " ").replace(" .", " ")
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'"-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def layer_2_dyslexia_fix(text, tokenizer, t5_model):
    if not tokenizer or not t5_model:
        return text
    try:
        input_text = "correct dyslexia: " + text
        inputs = tokenizer(input_text, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = t5_model.generate(**inputs, max_length=512, num_beams=2, early_stopping=True)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    except Exception as e:
        print(f"Error in Layer 2: {e}")
        return text

def layer_3_groq_correction(text, client):
    if not client:
        return text
    print("[NET] Contacting Groq (Llama 3.3)...")
    system_prompt = """
    You are an expert English Editor specializing in fixing OCR errors and dyslexic text.
    1. Fix spelling and grammatical errors based on context.
    2. Do NOT add conversational filler. Output ONLY the corrected text.
    """
    try:
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ],
            temperature=0.1,
            max_tokens=1024
        )
        return completion.choices[0].message.content.strip()
    except Exception as e:
        print(f"[ERROR] Groq API Error: {e}")
        return text

# --- Main Engine ---
def run_full_pipeline(image_path):
    print(f"\nProcessing image: {image_path}")
    
    detection_model, base_processor, base_model = load_ocr_models()
    tokenizer, t5_model = load_local_t5_model()
    client = initialize_groq_client()

    print("\n--- Running OCR (DocTR + TrOCR) ---")
    initial_ocr, boxed_img = run_ocr_inference(image_path, detection_model, base_processor, base_model)
    print(f"Initial OCR Paragraph:\n{initial_ocr}\n")

    print("--- Running Layer 1 (Sanitize) ---")
    sanitized_text = layer_1_sanitize(initial_ocr)
    print(f"Sanitized Text:\n{sanitized_text}\n")

    print("--- Running Layer 2 (Local Dyslexia Fix) ---")
    local_corrected_text = layer_2_dyslexia_fix(sanitized_text, tokenizer, t5_model)
    print(f"Local Corrected Text (T5 Model):\n{local_corrected_text}\n")

    print("--- Running Layer 3 (Groq Context Fix) ---")
    final_corrected_text = layer_3_groq_correction(local_corrected_text, client)
    print(f"Final Corrected Text (Groq API):\n{final_corrected_text}\n")

    print("--- Full Pipeline Execution Complete ---")
    
    # Save the output image instead of using plt.show() which blocks execution
    output_img_path = "output_boxed_" + os.path.basename(image_path)
    # Convert RGB back to BGR for cv2 saving
    boxed_img_bgr = cv2.cvtColor(boxed_img, cv2.COLOR_RGB2BGR)
    cv2.imwrite(output_img_path, boxed_img_bgr)
    print(f"Saved boxed image to {output_img_path}")
    
    return final_corrected_text


c:\Users\abuba\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu

Processing image: test_image.png
Loading DocTR line detector...
Loading BASE TrOCR Large Handwritten...


The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 635/635 [00:01<00:00, 423.34it/s, Materializing param=encoder.layernorm.weight]                                      
The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-large-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING

[OK] OCR Models loaded successfully
[WARN] Local zipped model 'DyslexAI_Best_Model (1).zip' not found. Skipping local T5 step.
[OK] Groq client initialized.

--- Running OCR (DocTR + TrOCR) ---
Initial OCR Paragraph:
if the threatened , counter , revolution " not # to bring the President , backs . thes . 13 Stakes , of the Commonwealth was an occasion worthy , of his presence after . all it was Mr. Nklymuh

--- Running Layer 1 (Sanitize) ---
Sanitized Text:
if the threatened , counter , revolution " not to bring the President , backs thes 13 Stakes , of the Commonwealth was an occasion worthy , of his presence after all it was Mr. Nklymuh

--- Running Layer 2 (Local Dyslexia Fix) ---
Local Corrected Text (T5 Model):
if the threatened , counter , revolution " not to bring the President , backs thes 13 Stakes , of the Commonwealth was an occasion worthy , of his presence after all it was Mr. Nklymuh

--- Running Layer 3 (Groq Context Fix) ---
[NET] Contacting Groq (Llama 3.3)...
Final Co

In [ ]:
import os
import re
import math
import json
import time
import copy
import hashlib
import logging
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Tuple, Optional, Dict, Any

import cv2
import numpy as np
import torch
from PIL import Image
from paddleocr import PaddleOCR
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)


# ============================================================================
# CPU-ONLY, FREE, HIGH-QUALITY DYSLEXIA OCR PIPELINE
# ----------------------------------------------------------------------------
# Goal
# -----
# Build a strong, production-style OCR application for difficult handwriting and
# noisy real-world images without fine-tuning and without GPU dependence.
#
# Core design principles
# ----------------------
# 1) Do not rely on one OCR engine for every case.
# 2) Use a fast primary OCR engine for all lines.
# 3) Use a stronger fallback OCR engine only for suspicious lines.
# 4) Try multiple rescue preprocessing variants for hard lines.
# 5) Score candidates conservatively instead of blindly trusting one model.
# 6) Preserve transparency by returning raw OCR, corrected OCR, and edit diffs.
# ============================================================================


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("dyslexia_ocr")


@dataclass
class ModelConfig:
    """Configuration for model names and generation settings."""

    paddle_lang: str = "en"
    paddle_use_angle_cls: bool = True
    trocr_model_name: str = "microsoft/trocr-large-handwritten"
    byt5_model_name: str = "google/byt5-small"
    max_correction_length: int = 512
    correction_prompt_prefix: str = (
        "Correct OCR and dyslexia-related spelling errors while preserving meaning: "
    )
    trocr_max_new_tokens: int = 128
    trocr_num_beams: int = 4
    correction_num_beams: int = 4


@dataclass
class RoutingConfig:
    """Thresholds and routing behavior for fallback and correction."""

    min_line_height: int = 12
    min_line_width: int = 20
    paddle_accept_confidence: float = 0.82
    paddle_fallback_confidence: float = 0.60
    max_weird_char_ratio: float = 0.20
    max_digit_ratio_for_sentence: float = 0.55
    use_trocr_fallback: bool = True
    use_correction: bool = True
    skip_correction_for_very_short_text: bool = False
    min_text_len_for_correction: int = 1
    max_fallback_workers: int = 2
    enable_crop_cache: bool = True
    enable_language_rescoring: bool = True
    quality_mode: str = "best"


@dataclass
class PreprocessConfig:
    """Configuration for adaptive image preprocessing."""

    enable_deskew: bool = True
    enable_denoise: bool = True
    enable_contrast: bool = True
    enable_adaptive_threshold: bool = True
    threshold_only_for_low_contrast: bool = True
    crop_margins: bool = True
    line_variant_include_soft_gray: bool = True
    line_variant_include_adaptive_thresh: bool = True
    line_variant_include_clahe: bool = True
    line_variant_include_otsu: bool = True


@dataclass
class AppConfig:
    """Top-level application configuration."""

    models: ModelConfig = field(default_factory=ModelConfig)
    routing: RoutingConfig = field(default_factory=RoutingConfig)
    preprocess: PreprocessConfig = field(default_factory=PreprocessConfig)
    save_debug_images: bool = True
    debug_dir: str = "debug_outputs"


@dataclass
class TriageResult:
    blur_score: float
    brightness: float
    contrast: float
    is_low_contrast: bool
    estimated_skew_angle: float
    should_threshold: bool
    should_deskew: bool


@dataclass
class OCRLine:
    """Represents one OCR line and all related routing / correction metadata."""

    bbox: Tuple[int, int, int, int]
    raw_text: str
    confidence: float
    source: str
    corrected_text: Optional[str] = None
    merged_text: Optional[str] = None
    fallback_used: bool = False
    suspicious: bool = False
    chosen_variant: Optional[str] = None
    candidate_scores: List[Dict[str, Any]] = field(default_factory=list)

    def final_text(self) -> str:
        return (self.corrected_text or self.merged_text or self.raw_text or "").strip()


@dataclass
class OCRResult:
    image_path: str
    raw_text: str
    corrected_text: str
    lines: List[OCRLine]
    triage: TriageResult
    metadata: Dict[str, Any]


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def clamp(v: int, lo: int, hi: int) -> int:
    return max(lo, min(v, hi))


def to_rgb(img_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def to_gray(img_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)


def pil_from_rgb(rgb: np.ndarray) -> Image.Image:
    return Image.fromarray(rgb)


def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def weird_char_ratio(text: str) -> float:
    """Rough signal for OCR noise based on unusual character usage."""
    if not text:
        return 1.0
    weird = len(re.findall(r"[^A-Za-z0-9\s.,!?;:'\"()\-]", text))
    return weird / max(1, len(text))


def digit_ratio(text: str) -> float:
    """Fraction of characters that are digits."""
    if not text:
        return 0.0
    digits = len(re.findall(r"\d", text))
    return digits / max(1, len(text))


def levenshtein_ops(a: str, b: str) -> List[Dict[str, Any]]:
    """SequenceMatcher-based diff summary for transparent UI output."""
    import difflib

    matcher = difflib.SequenceMatcher(a=a, b=b)
    ops: List[Dict[str, Any]] = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            ops.append(
                {
                    "op": tag,
                    "from": a[i1:i2],
                    "to": b[j1:j2],
                    "a_span": [i1, i2],
                    "b_span": [j1, j2],
                }
            )
    return ops


def hash_array(arr: np.ndarray) -> str:
    """Stable cache key for image crops."""
    return hashlib.md5(arr.tobytes()).hexdigest()


class ImageTriage:
    """Analyze image properties before OCR to drive adaptive preprocessing."""

    @staticmethod
    def estimate_skew(gray: np.ndarray) -> float:
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        lines = cv2.HoughLinesP(
            edges,
            1,
            np.pi / 180,
            threshold=80,
            minLineLength=max(30, gray.shape[1] // 8),
            maxLineGap=10,
        )
        if lines is None:
            return 0.0

        angles: List[float] = []
        for line in lines[:200]:
            x1, y1, x2, y2 = line[0]
            angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
            if -45 <= angle <= 45:
                angles.append(angle)
        if not angles:
            return 0.0
        return float(np.median(angles))

    @staticmethod
    def analyze(img_bgr: np.ndarray) -> TriageResult:
        gray = to_gray(img_bgr)
        blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        brightness = float(np.mean(gray))
        contrast = float(np.std(gray))
        skew = ImageTriage.estimate_skew(gray)

        is_low_contrast = contrast < 45
        should_threshold = is_low_contrast
        should_deskew = abs(skew) > 1.5

        return TriageResult(
            blur_score=blur_score,
            brightness=brightness,
            contrast=contrast,
            is_low_contrast=is_low_contrast,
            estimated_skew_angle=skew,
            should_threshold=should_threshold,
            should_deskew=should_deskew,
        )


class Preprocessor:
    """Main image preprocessing tuned for OCR rather than appearance."""

    def __init__(self, config: PreprocessConfig):
        self.config = config

    def deskew(self, img_bgr: np.ndarray, angle: float) -> np.ndarray:
        h, w = img_bgr.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        return cv2.warpAffine(
            img_bgr,
            M,
            (w, h),
            flags=cv2.INTER_CUBIC,
            borderMode=cv2.BORDER_REPLICATE,
        )

    def crop_borders(self, img_bgr: np.ndarray) -> np.ndarray:
        gray = to_gray(img_bgr)
        _, th = cv2.threshold(gray, 245, 255, cv2.THRESH_BINARY_INV)
        coords = cv2.findNonZero(th)
        if coords is None:
            return img_bgr

        x, y, w, h = cv2.boundingRect(coords)
        pad = 10
        x0 = clamp(x - pad, 0, img_bgr.shape[1] - 1)
        y0 = clamp(y - pad, 0, img_bgr.shape[0] - 1)
        x1 = clamp(x + w + pad, 1, img_bgr.shape[1])
        y1 = clamp(y + h + pad, 1, img_bgr.shape[0])
        return img_bgr[y0:y1, x0:x1]

    def process(self, img_bgr: np.ndarray, triage: TriageResult) -> np.ndarray:
        out = img_bgr.copy()

        if self.config.crop_margins:
            out = self.crop_borders(out)

        if self.config.enable_deskew and triage.should_deskew:
            out = self.deskew(out, triage.estimated_skew_angle)

        if self.config.enable_denoise:
            out = cv2.fastNlMeansDenoisingColored(out, None, 7, 7, 7, 21)

        if self.config.enable_contrast:
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            l = clahe.apply(l)
            out = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

        if self.config.enable_adaptive_threshold and (
            (not self.config.threshold_only_for_low_contrast) or triage.should_threshold
        ):
            gray = to_gray(out)
            th = cv2.adaptiveThreshold(
                gray,
                255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                31,
                15,
            )
            out = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)

        return out

    def build_line_variants(self, crop_bgr: np.ndarray) -> Dict[str, np.ndarray]:
        """Build multiple rescue variants for difficult handwritten line crops."""
        gray = to_gray(crop_bgr)
        variants: Dict[str, np.ndarray] = {}

        if self.config.line_variant_include_soft_gray:
            soft = cv2.GaussianBlur(gray, (3, 3), 0)
            variants["soft_gray"] = cv2.cvtColor(soft, cv2.COLOR_GRAY2BGR)
        else:
            soft = cv2.GaussianBlur(gray, (3, 3), 0)

        if self.config.line_variant_include_adaptive_thresh:
            th = cv2.adaptiveThreshold(
                soft,
                255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                21,
                11,
            )
            variants["adaptive_thresh"] = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)

        if self.config.line_variant_include_clahe:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
            boosted = clahe.apply(gray)
            variants["clahe"] = cv2.cvtColor(boosted, cv2.COLOR_GRAY2BGR)

        if self.config.line_variant_include_otsu:
            _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            variants["otsu"] = cv2.cvtColor(otsu, cv2.COLOR_GRAY2BGR)

        if not variants:
            variants["original"] = crop_bgr

        return variants


class PaddleEngine:
    """Primary OCR engine used on the full preprocessed image."""

    def __init__(self, config: ModelConfig):
        logger.info("Loading PaddleOCR on CPU...")
        # recent PaddleOCR releases (3.x+) no longer accept `use_gpu` or
        # `show_log`, and `use_angle_cls` has been deprecated in favor of
        # `use_textline_orientation`.  Build a kwargs dict so we stay
        # compatible across versions and avoid passing unknown arguments.
        paddle_args: Dict[str, Any] = {}

        # map the old angle classifier flag to the new textline orientation
        # parameter; if the config doesn't expose the old field we simply
        # omit it and let the library choose a sensible default.
        if getattr(config, "paddle_use_angle_cls", False):
            paddle_args["use_textline_orientation"] = True

        # language is still a valid argument and we forward it if set.
        if getattr(config, "paddle_lang", None):
            paddle_args["lang"] = config.paddle_lang

        # other configuration options can be added here in the future
        # without triggering errors in newer PaddleOCR releases.
        self.ocr = PaddleOCR(**paddle_args)

    def run(self, img_bgr: np.ndarray) -> List[OCRLine]:
        result = self.ocr.ocr(img_bgr, cls=True)
        lines: List[OCRLine] = []

        if not result or not result[0]:
            return lines

        for item in result[0]:
            points, (text, conf) = item
            xs = [int(p[0]) for p in points]
            ys = [int(p[1]) for p in points]
            x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
            lines.append(
                OCRLine(
                    bbox=(x0, y0, x1, y1),
                    raw_text=text or "",
                    confidence=float(conf),
                    source="paddle",
                )
            )

        lines.sort(key=lambda r: (r.bbox[1], r.bbox[0]))
        return lines


class TrOCREngine:
    """Heavy handwritten fallback recognizer for suspicious line crops."""

    def __init__(self, config: ModelConfig):
        logger.info("Loading TrOCR fallback model on CPU...")
        self.processor = TrOCRProcessor.from_pretrained(config.trocr_model_name)
        self.model = VisionEncoderDecoderModel.from_pretrained(config.trocr_model_name)
        self.model.to("cpu")
        self.model.eval()
        self.max_new_tokens = config.trocr_max_new_tokens
        self.num_beams = config.trocr_num_beams

    def recognize_crop(self, crop_bgr: np.ndarray) -> str:
        crop_rgb = to_rgb(crop_bgr)
        pil_img = pil_from_rgb(crop_rgb)
        pixel_values = self.processor(images=pil_img, return_tensors="pt").pixel_values

        with torch.no_grad():
            generated_ids = self.model.generate(
                pixel_values,
                max_new_tokens=self.max_new_tokens,
                num_beams=self.num_beams,
                early_stopping=True,
            )
        text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return normalize_spaces(text)


class ByT5Corrector:
    """Text correction stage separated from OCR for cleaner error repair."""

    def __init__(self, config: ModelConfig):
        logger.info("Loading ByT5 correction model on CPU...")
        self.tokenizer = AutoTokenizer.from_pretrained(config.byt5_model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(config.byt5_model_name)
        self.model.to("cpu")
        self.model.eval()
        self.max_len = config.max_correction_length
        self.prefix = config.correction_prompt_prefix
        self.num_beams = config.correction_num_beams

    def correct(self, text: str) -> str:
        if not text.strip():
            return text

        prompt = self.prefix + text
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=self.max_len,
                num_beams=self.num_beams,
                early_stopping=True,
            )
        out = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return normalize_spaces(out)


class Router:
    """Decides which lines are suspicious enough to deserve fallback OCR."""

    def __init__(self, config: RoutingConfig):
        self.config = config

    def is_suspicious(self, line: OCRLine) -> bool:
        txt = line.raw_text or ""
        if not txt.strip():
            return True

        if line.confidence < self.config.paddle_fallback_confidence:
            return True

        if weird_char_ratio(txt) > self.config.max_weird_char_ratio:
            return True

        if len(txt) >= 8 and digit_ratio(txt) > self.config.max_digit_ratio_for_sentence:
            return True

        return False

    def needs_fallback(self, line: OCRLine) -> bool:
        if line.confidence >= self.config.paddle_accept_confidence and not self.is_suspicious(line):
            return False
        return self.is_suspicious(line)


class FusionEngine:
    """Conservative scorer for resolving disagreements between OCR engines."""

    COMMON_WORDS = {
        "the", "and", "is", "are", "was", "were", "this", "that", "with", "for",
        "have", "has", "you", "your", "from", "will", "can", "not", "text", "image",
        "school", "student", "name", "paragraph", "line", "note", "handwriting",
        "correct", "correction", "word", "sentence", "book", "page", "read",
    }

    @staticmethod
    def language_plausibility_score(text: str) -> float:
        text = normalize_spaces(text)
        if not text:
            return -1e9

        weird_penalty = weird_char_ratio(text) * 6.0
        digit_penalty = max(0.0, digit_ratio(text) - 0.25) * 4.0

        tokens = re.findall(r"[A-Za-z']+", text.lower())
        if not tokens:
            token_score = -2.0
        else:
            common_hits = sum(1 for t in tokens if t in FusionEngine.COMMON_WORDS)
            avg_token_len = sum(len(t) for t in tokens) / max(1, len(tokens))
            token_score = (common_hits / max(1, len(tokens))) * 2.5
            token_score += min(avg_token_len / 6.0, 1.0)

        punctuation_balance = 0.0
        if text.count("(") == text.count(")"):
            punctuation_balance += 0.2
        if text.count('"') % 2 == 0:
            punctuation_balance += 0.1

        length_bonus = min(len(text) / 80.0, 1.0)
        return token_score + punctuation_balance + length_bonus - weird_penalty - digit_penalty

    @staticmethod
    def choose_text(paddle_text: str, trocr_text: str) -> str:
        pt = normalize_spaces(paddle_text)
        tt = normalize_spaces(trocr_text)

        if not tt:
            return pt
        if not pt:
            return tt

        pt_score = FusionEngine.language_plausibility_score(pt)
        tt_score = FusionEngine.language_plausibility_score(tt)

        if tt_score > pt_score + 0.20:
            return tt
        return pt


class DyslexiaOCRApp:
    """Main OCR application with quality modes, fallback routing, and exports."""

    def __init__(self, config: Optional[AppConfig] = None):
        self.config = copy.deepcopy(config) if config is not None else AppConfig()
        ensure_dir(self.config.debug_dir)

        self._apply_quality_mode()

        self.preprocessor = Preprocessor(self.config.preprocess)
        self.router = Router(self.config.routing)
        self.fusion = FusionEngine()

        self.paddle = PaddleEngine(self.config.models)
        self.trocr = TrOCREngine(self.config.models) if self.config.routing.use_trocr_fallback else None
        self.corrector = ByT5Corrector(self.config.models) if self.config.routing.use_correction else None
        self.crop_cache: Dict[str, str] = {}

    def _apply_quality_mode(self) -> None:
        """Configure thresholds and worker counts based on quality mode."""
        mode = (self.config.routing.quality_mode or "best").lower()
        if mode == "fast":
            self.config.routing.paddle_accept_confidence = 0.78
            self.config.routing.paddle_fallback_confidence = 0.55
            self.config.routing.max_fallback_workers = 1
        elif mode == "balanced":
            self.config.routing.paddle_accept_confidence = 0.82
            self.config.routing.paddle_fallback_confidence = 0.60
            self.config.routing.max_fallback_workers = 2
        else:
            self.config.routing.paddle_accept_confidence = 0.88
            self.config.routing.paddle_fallback_confidence = 0.70
            self.config.routing.max_fallback_workers = 3

    def _save_debug_image(self, name: str, img_bgr: np.ndarray) -> None:
        if not self.config.save_debug_images:
            return
        path = os.path.join(self.config.debug_dir, name)
        cv2.imwrite(path, img_bgr)

    def _crop_line(self, img_bgr: np.ndarray, bbox: Tuple[int, int, int, int], pad: int = 6) -> np.ndarray:
        x0, y0, x1, y1 = bbox
        h, w = img_bgr.shape[:2]
        x0 = clamp(x0 - pad, 0, w - 1)
        y0 = clamp(y0 - pad, 0, h - 1)
        x1 = clamp(x1 + pad, 1, w)
        y1 = clamp(y1 + pad, 1, h)
        return img_bgr[y0:y1, x0:x1]

    def _process_fallback_line(self, preprocessed: np.ndarray, line: OCRLine, idx: int) -> OCRLine:
        """Process one suspicious line through multi-pass TrOCR fallback."""
        if not self.trocr:
            return line

        crop = self._crop_line(preprocessed, line.bbox)
        crop_key = hash_array(crop)

        if self.config.routing.enable_crop_cache and crop_key in self.crop_cache:
            cached_text = self.crop_cache[crop_key]
            line.fallback_used = True
            line.merged_text = self.fusion.choose_text(line.raw_text, cached_text)
            line.chosen_variant = "cache"
            line.candidate_scores.append(
                {
                    "variant": "cache",
                    "trocr_text": cached_text,
                    "fused": line.merged_text,
                    "score": self.fusion.language_plausibility_score(line.merged_text),
                }
            )
            return line

        variants = self.preprocessor.build_line_variants(crop)
        candidates: List[Tuple[str, str, str, float]] = []

        for variant_name, variant_img in variants.items():
            trocr_text = self.trocr.recognize_crop(variant_img)
            fused = self.fusion.choose_text(line.raw_text, trocr_text)
            score = self.fusion.language_plausibility_score(fused)
            line.candidate_scores.append(
                {
                    "variant": variant_name,
                    "trocr_text": trocr_text,
                    "fused": fused,
                    "score": score,
                }
            )
            candidates.append((variant_name, trocr_text, fused, score))

        best_variant_name, best_trocr_text, best_fused, best_score = max(candidates, key=lambda item: item[3])
        line.fallback_used = True
        line.merged_text = best_fused
        line.chosen_variant = best_variant_name

        if self.config.routing.enable_crop_cache:
            self.crop_cache[crop_key] = best_trocr_text

        logger.info(
            "Line %d used TrOCR fallback via variant '%s' with score %.3f",
            idx,
            best_variant_name,
            best_score,
        )
        return line

    def _correct_lines(self, lines: List[OCRLine]) -> str:
        """Apply text correction line by line for stability."""
        outputs: List[str] = []
        for line in lines:
            base_text = normalize_spaces(line.merged_text or line.raw_text)
            if not self.corrector:
                line.corrected_text = base_text
            else:
                if (
                    self.config.routing.skip_correction_for_very_short_text
                    and len(base_text) < self.config.routing.min_text_len_for_correction
                ):
                    line.corrected_text = base_text
                else:
                    line.corrected_text = self.corrector.correct(base_text)
            outputs.append(line.corrected_text)
        return normalize_spaces(" ".join(outputs))

    def process_image(self, image_path: str) -> OCRResult:
        """End-to-end OCR pipeline for a single image."""
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            raise ValueError(f"Could not read image: {image_path}")

        stage_times: Dict[str, float] = {}
        t_global = time.time()

        t0 = time.time()
        triage = ImageTriage.analyze(img_bgr)
        stage_times["triage_seconds"] = round(time.time() - t0, 4)

        t0 = time.time()
        preprocessed = self.preprocessor.process(img_bgr, triage)
        self._save_debug_image("01_preprocessed.png", preprocessed)
        stage_times["preprocess_seconds"] = round(time.time() - t0, 4)

        t0 = time.time()
        lines = self.paddle.run(preprocessed)
        stage_times["paddle_seconds"] = round(time.time() - t0, 4)

        fallback_jobs: List[Tuple[int, OCRLine]] = []
        for i, line in enumerate(lines, start=1):
            line.suspicious = self.router.is_suspicious(line)
            line.merged_text = line.raw_text

            x0, y0, x1, y1 = line.bbox
            if (x1 - x0) < self.config.routing.min_line_width or (y1 - y0) < self.config.routing.min_line_height:
                continue

            if self.trocr and self.router.needs_fallback(line):
                fallback_jobs.append((i, line))

        t0 = time.time()
        if self.trocr and fallback_jobs:
            with ThreadPoolExecutor(max_workers=self.config.routing.max_fallback_workers) as executor:
                future_map = {
                    executor.submit(self._process_fallback_line, preprocessed, line, idx): (idx, line)
                    for idx, line in fallback_jobs
                }
                for future in as_completed(future_map):
                    idx, original_line = future_map[future]
                    try:
                        processed_line = future.result()
                        original_line.merged_text = processed_line.merged_text
                        original_line.fallback_used = processed_line.fallback_used
                        original_line.chosen_variant = processed_line.chosen_variant
                        original_line.candidate_scores = processed_line.candidate_scores
                    except Exception as exc:
                        logger.warning("Parallel fallback failed on line %d: %s", idx, exc)
        stage_times["fallback_seconds"] = round(time.time() - t0, 4)

        t0 = time.time()
        raw_text = normalize_spaces(
            " ".join(
                (ln.merged_text or ln.raw_text)
                for ln in lines
                if (ln.merged_text or ln.raw_text).strip()
            )
        )
        corrected_text = self._correct_lines(lines)
        stage_times["correction_seconds"] = round(time.time() - t0, 4)

        runtime = time.time() - t_global
        metadata = {
            "runtime_seconds": round(runtime, 3),
            "num_lines": len(lines),
            "fallback_lines": sum(1 for ln in lines if ln.fallback_used),
            "suspicious_lines": sum(1 for ln in lines if ln.suspicious),
            "quality_mode": self.config.routing.quality_mode,
            "cache_entries": len(self.crop_cache),
            **stage_times,
        }

        return OCRResult(
            image_path=image_path,
            raw_text=raw_text,
            corrected_text=corrected_text,
            lines=lines,
            triage=triage,
            metadata=metadata,
        )

    def export_result_json(self, result: OCRResult, output_path: str) -> None:
        """Export a rich JSON result for debugging, UI integration, or audits."""
        payload = {
            "image_path": result.image_path,
            "raw_text": result.raw_text,
            "corrected_text": result.corrected_text,
            "triage": result.triage.__dict__,
            "metadata": result.metadata,
            "lines": [
                {
                    "bbox": line.bbox,
                    "raw_text": line.raw_text,
                    "confidence": line.confidence,
                    "source": line.source,
                    "merged_text": line.merged_text,
                    "corrected_text": line.corrected_text,
                    "fallback_used": line.fallback_used,
                    "suspicious": line.suspicious,
                    "chosen_variant": line.chosen_variant,
                    "candidate_scores": line.candidate_scores,
                    "edit_ops": levenshtein_ops(line.merged_text or line.raw_text, line.corrected_text or ""),
                }
                for line in result.lines
            ],
        }
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)


def _run_self_tests() -> None:
    """Lightweight self-tests for core pure functions and CLI-safe formatting."""
    assert normalize_spaces("a   b\n c") == "a b c"
    assert weird_char_ratio("") == 1.0
    assert digit_ratio("abc123") == 0.5

    ops = levenshtein_ops("cat", "coat")
    assert isinstance(ops, list)
    assert len(ops) >= 1

    score_clean = FusionEngine.language_plausibility_score("this is a clean sentence")
    score_noisy = FusionEngine.language_plausibility_score("%%% 9999 ???")
    assert score_clean > score_noisy

    sample_meta = {"runtime_seconds": 1.23}
    raw_block = "\n===== RAW OCR =====\n"
    corrected_block = "\n===== CORRECTED OCR =====\n"
    meta_block = "\n===== METADATA =====\n"
    assert raw_block.startswith("\n") and raw_block.endswith("\n")
    assert corrected_block.startswith("\n") and corrected_block.endswith("\n")
    assert meta_block.startswith("\n") and meta_block.endswith("\n")
    assert json.dumps(sample_meta, indent=2).startswith("{")

    print("Self-tests passed.")


def main() -> None:
    import argparse

    parser = argparse.ArgumentParser(description="CPU-only dyslexia OCR pipeline")
    parser.add_argument("image", nargs="?", help="Path to input image")
    parser.add_argument("--json-out", default="ocr_result.json", help="Output JSON path")
    parser.add_argument("--no-debug-images", action="store_true", help="Disable debug image saving")
    parser.add_argument(
        "--quality-mode",
        default="best",
        choices=["fast", "balanced", "best"],
        help="Pipeline quality mode",
    )
    parser.add_argument(
        "--run-self-tests",
        action="store_true",
        help="Run lightweight internal tests and exit",
    )
    args, _ = parser.parse_known_args()

    if args.run_self_tests:
        _run_self_tests()
        return

    if not args.image:
        parser.error("the following arguments are required: image (unless --run-self-tests is used)")

    config = AppConfig(save_debug_images=not args.no_debug_images)
    config.routing.quality_mode = args.quality_mode

    app = DyslexiaOCRApp(config)
    result = app.process_image(args.image)
    app.export_result_json(result, args.json_out)

    print("\n===== RAW OCR =====\n")
    print(result.raw_text)
    print("\n===== CORRECTED OCR =====\n")
    print(result.corrected_text)
    print("\n===== METADATA =====\n")
    print(json.dumps(result.metadata, indent=2))
    print(f"\nSaved JSON to: {args.json_out}")


ModuleNotFoundError: No module named 'paddleocr'

In [2]:
pip install paddleocr paddlepaddle

  Using cached paddleocr-3.4.0-py3-none-any.whl.metadata (55 kB)
  Using cached paddlepaddle-3.3.0-cp312-cp312-win_amd64.whl.metadata (8.8 kB)
  Using cached paddlex-3.4.2-py3-none-any.whl.metadata (80 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached aistudio_sdk-0.3.8-py3-none-any.whl.metadata (1.1 kB)
  Using cached chardet-6.0.0.post1-py3-none-any.whl.metadata (3.3 kB)
  Using cached colorlog-6.10.1-py3-none-any.whl.metadata (11 kB)
  Using cached modelscope-1.34.0-py3-none-any.whl.metadata (43 kB)
  Using cached imagesize-1.4.1-py2.py3-none-any.whl.metadata (1.5 kB)
  Using cached opencv_contrib_python-4.10.0.84-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached charset_normalizer-3.4.4-cp312-cp312-win_amd64.whl.metadata (38 kB)
  Using cached bce_python_sdk-0.9.60-py3-none-any.whl.metadata (416 bytes)
  Using cached future-1.0.0-py3-none-any.whl.metadata (4.0 kB)
Using cached paddleocr-3.4.0-py3-none-any.whl (87 kB)
Using cached paddlepaddl

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'c:\\Users\\abuba\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\cv2\\cv2.pyd'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import re
import math
import json
import time
import copy
import hashlib
import logging
from dataclasses import dataclass, field
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Tuple, Optional, Dict, Any

import cv2
import numpy as np
import torch
from PIL import Image

try:
    from paddleocr import PaddleOCR  # type: ignore
except ImportError:
    PaddleOCR = None

from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)


# ============================================================================
# CPU-ONLY, FREE, HIGH-QUALITY DYSLEXIA OCR PIPELINE
# ============================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("dyslexia_ocr")


# ============================================================================
# CONFIG
# ============================================================================

@dataclass
class ModelConfig:
    paddle_lang: str = "en"
    paddle_use_angle_cls: bool = True
    trocr_model_name: str = "microsoft/trocr-large-handwritten"
    byt5_model_name: str = "google/byt5-small"
    max_correction_length: int = 512
    correction_prompt_prefix: str = (
        "Correct OCR and dyslexia-related spelling errors while preserving meaning: "
    )
    trocr_max_new_tokens: int = 128
    trocr_num_beams: int = 4
    correction_num_beams: int = 4


@dataclass
class RoutingConfig:
    min_line_height: int = 12
    min_line_width: int = 20
    paddle_accept_confidence: float = 0.82
    paddle_fallback_confidence: float = 0.60
    max_weird_char_ratio: float = 0.20
    max_digit_ratio_for_sentence: float = 0.55
    use_trocr_fallback: bool = True
    use_correction: bool = True
    skip_correction_for_very_short_text: bool = False
    min_text_len_for_correction: int = 1
    max_fallback_workers: int = 2
    enable_crop_cache: bool = True
    enable_language_rescoring: bool = True
    quality_mode: str = "best"   # fast | balanced | best


@dataclass
class PreprocessConfig:
    enable_deskew: bool = True
    enable_denoise: bool = True
    enable_contrast: bool = True
    enable_adaptive_threshold: bool = True
    threshold_only_for_low_contrast: bool = True
    crop_margins: bool = True
    line_variant_include_soft_gray: bool = True
    line_variant_include_adaptive_thresh: bool = True
    line_variant_include_clahe: bool = True
    line_variant_include_otsu: bool = True


@dataclass
class AppConfig:
    models: ModelConfig = field(default_factory=ModelConfig)
    routing: RoutingConfig = field(default_factory=RoutingConfig)
    preprocess: PreprocessConfig = field(default_factory=PreprocessConfig)
    save_debug_images: bool = True
    debug_dir: str = "debug_outputs"


# ============================================================================
# DATA STRUCTURES
# ============================================================================

@dataclass
class TriageResult:
    blur_score: float
    brightness: float
    contrast: float
    is_low_contrast: bool
    estimated_skew_angle: float
    should_threshold: bool
    should_deskew: bool


@dataclass
class OCRLine:
    bbox: Tuple[int, int, int, int]
    raw_text: str
    confidence: float
    source: str
    corrected_text: Optional[str] = None
    merged_text: Optional[str] = None
    fallback_used: bool = False
    suspicious: bool = False
    chosen_variant: Optional[str] = None
    candidate_scores: List[Dict[str, Any]] = field(default_factory=list)

    def final_text(self) -> str:
        return (self.corrected_text or self.merged_text or self.raw_text or "").strip()


@dataclass
class OCRResult:
    image_path: str
    raw_text: str
    corrected_text: str
    lines: List[OCRLine]
    triage: TriageResult
    metadata: Dict[str, Any]


# ============================================================================
# HELPERS
# ============================================================================

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def clamp(v: int, lo: int, hi: int) -> int:
    return max(lo, min(v, hi))


def to_rgb(img_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def to_gray(img_bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)


def pil_from_rgb(rgb: np.ndarray) -> Image.Image:
    return Image.fromarray(rgb)


def normalize_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def weird_char_ratio(text: str) -> float:
    if not text:
        return 1.0
    weird = len(re.findall(r"[^A-Za-z0-9\s.,!?;:'\"()\-]", text))
    return weird / max(1, len(text))


def digit_ratio(text: str) -> float:
    if not text:
        return 0.0
    digits = len(re.findall(r"\d", text))
    return digits / max(1, len(text))


def levenshtein_ops(a: str, b: str) -> List[Dict[str, Any]]:
    import difflib

    matcher = difflib.SequenceMatcher(a=a, b=b)
    ops: List[Dict[str, Any]] = []
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            ops.append(
                {
                    "op": tag,
                    "from": a[i1:i2],
                    "to": b[j1:j2],
                    "a_span": [i1, i2],
                    "b_span": [j1, j2],
                }
            )
    return ops


def hash_array(arr: np.ndarray) -> str:
    return hashlib.md5(arr.tobytes()).hexdigest()


# ============================================================================
# TRIAGE
# ============================================================================

class ImageTriage:
    @staticmethod
    def estimate_skew(gray: np.ndarray) -> float:
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        lines = cv2.HoughLinesP(
            edges,
            1,
            np.pi / 180,
            threshold=80,
            minLineLength=max(30, gray.shape[1] // 8),
            maxLineGap=10,
        )
        if lines is None:
            return 0.0

        angles: List[float] = []
        for line in lines[:200]:
            x1, y1, x2, y2 = line[0]
            angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
            if -45 <= angle <= 45:
                angles.append(angle)

        if not angles:
            return 0.0
        return float(np.median(angles))

    @staticmethod
    def analyze(img_bgr: np.ndarray) -> TriageResult:
        gray = to_gray(img_bgr)
        blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        brightness = float(np.mean(gray))
        contrast = float(np.std(gray))
        skew = ImageTriage.estimate_skew(gray)

        is_low_contrast = contrast < 45
        should_threshold = is_low_contrast
        should_deskew = abs(skew) > 1.5

        return TriageResult(
            blur_score=blur_score,
            brightness=brightness,
            contrast=contrast,
            is_low_contrast=is_low_contrast,
            estimated_skew_angle=skew,
            should_threshold=should_threshold,
            should_deskew=should_deskew,
        )


# ============================================================================
# PREPROCESSING
# ============================================================================

class Preprocessor:
    def __init__(self, config: PreprocessConfig):
        self.config = config

    def deskew(self, img_bgr: np.ndarray, angle: float) -> np.ndarray:
        h, w = img_bgr.shape[:2]
        center = (w // 2, h // 2)
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        return cv2.warpAffine(
            img_bgr,
            M,
            (w, h),
            flags=cv2.INTER_CUBIC,
            borderMode=cv2.BORDER_REPLICATE,
        )

    def crop_borders(self, img_bgr: np.ndarray) -> np.ndarray:
        gray = to_gray(img_bgr)
        _, th = cv2.threshold(gray, 245, 255, cv2.THRESH_BINARY_INV)
        coords = cv2.findNonZero(th)
        if coords is None:
            return img_bgr

        x, y, w, h = cv2.boundingRect(coords)
        pad = 10
        x0 = clamp(x - pad, 0, img_bgr.shape[1] - 1)
        y0 = clamp(y - pad, 0, img_bgr.shape[0] - 1)
        x1 = clamp(x + w + pad, 1, img_bgr.shape[1])
        y1 = clamp(y + h + pad, 1, img_bgr.shape[0])
        return img_bgr[y0:y1, x0:x1]

    def process(self, img_bgr: np.ndarray, triage: TriageResult) -> np.ndarray:
        out = img_bgr.copy()

        if self.config.crop_margins:
            out = self.crop_borders(out)

        if self.config.enable_deskew and triage.should_deskew:
            out = self.deskew(out, triage.estimated_skew_angle)

        if self.config.enable_denoise:
            out = cv2.fastNlMeansDenoisingColored(out, None, 7, 7, 7, 21)

        if self.config.enable_contrast:
            lab = cv2.cvtColor(out, cv2.COLOR_BGR2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            l = clahe.apply(l)
            out = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)

        if self.config.enable_adaptive_threshold and (
            (not self.config.threshold_only_for_low_contrast) or triage.should_threshold
        ):
            gray = to_gray(out)
            th = cv2.adaptiveThreshold(
                gray,
                255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                31,
                15,
            )
            out = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)

        return out

    def build_line_variants(self, crop_bgr: np.ndarray) -> Dict[str, np.ndarray]:
        gray = to_gray(crop_bgr)
        variants: Dict[str, np.ndarray] = {}

        if self.config.line_variant_include_soft_gray:
            soft = cv2.GaussianBlur(gray, (3, 3), 0)
            variants["soft_gray"] = cv2.cvtColor(soft, cv2.COLOR_GRAY2BGR)
        else:
            soft = cv2.GaussianBlur(gray, (3, 3), 0)

        if self.config.line_variant_include_adaptive_thresh:
            th = cv2.adaptiveThreshold(
                soft,
                255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                21,
                11,
            )
            variants["adaptive_thresh"] = cv2.cvtColor(th, cv2.COLOR_GRAY2BGR)

        if self.config.line_variant_include_clahe:
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
            boosted = clahe.apply(gray)
            variants["clahe"] = cv2.cvtColor(boosted, cv2.COLOR_GRAY2BGR)

        if self.config.line_variant_include_otsu:
            _, otsu = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            variants["otsu"] = cv2.cvtColor(otsu, cv2.COLOR_GRAY2BGR)

        if not variants:
            variants["original"] = crop_bgr

        return variants


# ============================================================================
# OCR ENGINES
# ============================================================================

class PaddleEngine:
    def __init__(self, config: ModelConfig):
        if PaddleOCR is None:
            raise ImportError(
                "PaddleOCR is not installed. Install it with: pip install paddleocr paddlepaddle"
            )

        logger.info("Loading PaddleOCR on CPU...")
        self.ocr = PaddleOCR(
            use_angle_cls=config.paddle_use_angle_cls,
            lang=config.paddle_lang,
            use_gpu=False,
            show_log=False,
        )

    def run(self, img_bgr: np.ndarray) -> List[OCRLine]:
        result = self.ocr.ocr(img_bgr, cls=True)
        lines: List[OCRLine] = []

        if not result or not result[0]:
            return lines

        for item in result[0]:
            points, (text, conf) = item
            xs = [int(p[0]) for p in points]
            ys = [int(p[1]) for p in points]
            x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
            lines.append(
                OCRLine(
                    bbox=(x0, y0, x1, y1),
                    raw_text=text or "",
                    confidence=float(conf),
                    source="paddle",
                )
            )

        lines.sort(key=lambda r: (r.bbox[1], r.bbox[0]))
        return lines


class TrOCREngine:
    def __init__(self, config: ModelConfig):
        logger.info("Loading TrOCR fallback model on CPU...")
        self.processor = TrOCRProcessor.from_pretrained(config.trocr_model_name)
        self.model = VisionEncoderDecoderModel.from_pretrained(config.trocr_model_name)
        self.model.to("cpu")
        self.model.eval()
        self.max_new_tokens = config.trocr_max_new_tokens
        self.num_beams = config.trocr_num_beams

    def recognize_crop(self, crop_bgr: np.ndarray) -> str:
        crop_rgb = to_rgb(crop_bgr)
        pil_img = pil_from_rgb(crop_rgb)
        pixel_values = self.processor(images=pil_img, return_tensors="pt").pixel_values

        with torch.no_grad():
            generated_ids = self.model.generate(
                pixel_values,
                max_new_tokens=self.max_new_tokens,
                num_beams=self.num_beams,
                early_stopping=True,
            )
        text = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        return normalize_spaces(text)


class ByT5Corrector:
    def __init__(self, config: ModelConfig):
        logger.info("Loading ByT5 correction model on CPU...")
        self.tokenizer = AutoTokenizer.from_pretrained(config.byt5_model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(config.byt5_model_name)
        self.model.to("cpu")
        self.model.eval()
        self.max_len = config.max_correction_length
        self.prefix = config.correction_prompt_prefix
        self.num_beams = config.correction_num_beams

    def correct(self, text: str) -> str:
        if not text.strip():
            return text

        prompt = self.prefix + text
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_length=self.max_len,
                num_beams=self.num_beams,
                early_stopping=True,
            )
        out = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return normalize_spaces(out)


# ============================================================================
# ROUTING / FUSION
# ============================================================================

class Router:
    def __init__(self, config: RoutingConfig):
        self.config = config

    def is_suspicious(self, line: OCRLine) -> bool:
        txt = line.raw_text or ""
        if not txt.strip():
            return True

        if line.confidence < self.config.paddle_fallback_confidence:
            return True

        if weird_char_ratio(txt) > self.config.max_weird_char_ratio:
            return True

        if len(txt) >= 8 and digit_ratio(txt) > self.config.max_digit_ratio_for_sentence:
            return True

        return False

    def needs_fallback(self, line: OCRLine) -> bool:
        if line.confidence >= self.config.paddle_accept_confidence and not self.is_suspicious(line):
            return False
        return self.is_suspicious(line)


class FusionEngine:
    COMMON_WORDS = {
        "the", "and", "is", "are", "was", "were", "this", "that", "with", "for",
        "have", "has", "you", "your", "from", "will", "can", "not", "text", "image",
        "school", "student", "name", "paragraph", "line", "note", "handwriting",
        "correct", "correction", "word", "sentence", "book", "page", "read",
    }

    @staticmethod
    def language_plausibility_score(text: str) -> float:
        text = normalize_spaces(text)
        if not text:
            return -1e9

        weird_penalty = weird_char_ratio(text) * 6.0
        digit_penalty = max(0.0, digit_ratio(text) - 0.25) * 4.0

        tokens = re.findall(r"[A-Za-z']+", text.lower())
        if not tokens:
            token_score = -2.0
        else:
            common_hits = sum(1 for t in tokens if t in FusionEngine.COMMON_WORDS)
            avg_token_len = sum(len(t) for t in tokens) / max(1, len(tokens))
            token_score = (common_hits / max(1, len(tokens))) * 2.5
            token_score += min(avg_token_len / 6.0, 1.0)

        punctuation_balance = 0.0
        if text.count("(") == text.count(")"):
            punctuation_balance += 0.2
        if text.count('"') % 2 == 0:
            punctuation_balance += 0.1

        length_bonus = min(len(text) / 80.0, 1.0)
        return token_score + punctuation_balance + length_bonus - weird_penalty - digit_penalty

    @staticmethod
    def choose_text(paddle_text: str, trocr_text: str) -> str:
        pt = normalize_spaces(paddle_text)
        tt = normalize_spaces(trocr_text)

        if not tt:
            return pt
        if not pt:
            return tt

        pt_score = FusionEngine.language_plausibility_score(pt)
        tt_score = FusionEngine.language_plausibility_score(tt)

        if tt_score > pt_score + 0.20:
            return tt
        return pt


# ============================================================================
# MAIN APP
# ============================================================================

class DyslexiaOCRApp:
    def __init__(self, config: Optional[AppConfig] = None):
        self.config = copy.deepcopy(config) if config is not None else AppConfig()
        ensure_dir(self.config.debug_dir)

        self._apply_quality_mode()

        self.preprocessor = Preprocessor(self.config.preprocess)
        self.router = Router(self.config.routing)
        self.fusion = FusionEngine()

        self.paddle = PaddleEngine(self.config.models)
        self.trocr = TrOCREngine(self.config.models) if self.config.routing.use_trocr_fallback else None
        self.corrector = ByT5Corrector(self.config.models) if self.config.routing.use_correction else None
        self.crop_cache: Dict[str, str] = {}

    def _apply_quality_mode(self) -> None:
        mode = (self.config.routing.quality_mode or "best").lower()
        if mode == "fast":
            self.config.routing.paddle_accept_confidence = 0.78
            self.config.routing.paddle_fallback_confidence = 0.55
            self.config.routing.max_fallback_workers = 1
        elif mode == "balanced":
            self.config.routing.paddle_accept_confidence = 0.82
            self.config.routing.paddle_fallback_confidence = 0.60
            self.config.routing.max_fallback_workers = 2
        else:
            self.config.routing.paddle_accept_confidence = 0.88
            self.config.routing.paddle_fallback_confidence = 0.70
            self.config.routing.max_fallback_workers = 3

    def _save_debug_image(self, name: str, img_bgr: np.ndarray) -> None:
        if not self.config.save_debug_images:
            return
        path = os.path.join(self.config.debug_dir, name)
        cv2.imwrite(path, img_bgr)

    def _crop_line(self, img_bgr: np.ndarray, bbox: Tuple[int, int, int, int], pad: int = 6) -> np.ndarray:
        x0, y0, x1, y1 = bbox
        h, w = img_bgr.shape[:2]
        x0 = clamp(x0 - pad, 0, w - 1)
        y0 = clamp(y0 - pad, 0, h - 1)
        x1 = clamp(x1 + pad, 1, w)
        y1 = clamp(y1 + pad, 1, h)
        return img_bgr[y0:y1, x0:x1]

    def _process_fallback_line(self, preprocessed: np.ndarray, line: OCRLine, idx: int) -> OCRLine:
        if not self.trocr:
            return line

        crop = self._crop_line(preprocessed, line.bbox)
        crop_key = hash_array(crop)

        if self.config.routing.enable_crop_cache and crop_key in self.crop_cache:
            cached_text = self.crop_cache[crop_key]
            line.fallback_used = True
            line.merged_text = self.fusion.choose_text(line.raw_text, cached_text)
            line.chosen_variant = "cache"
            line.candidate_scores.append(
                {
                    "variant": "cache",
                    "trocr_text": cached_text,
                    "fused": line.merged_text,
                    "score": self.fusion.language_plausibility_score(line.merged_text),
                }
            )
            return line

        variants = self.preprocessor.build_line_variants(crop)
        candidates: List[Tuple[str, str, str, float]] = []

        for variant_name, variant_img in variants.items():
            trocr_text = self.trocr.recognize_crop(variant_img)
            fused = self.fusion.choose_text(line.raw_text, trocr_text)
            score = self.fusion.language_plausibility_score(fused)
            line.candidate_scores.append(
                {
                    "variant": variant_name,
                    "trocr_text": trocr_text,
                    "fused": fused,
                    "score": score,
                }
            )
            candidates.append((variant_name, trocr_text, fused, score))

        best_variant_name, best_trocr_text, best_fused, best_score = max(candidates, key=lambda item: item[3])
        line.fallback_used = True
        line.merged_text = best_fused
        line.chosen_variant = best_variant_name

        if self.config.routing.enable_crop_cache:
            self.crop_cache[crop_key] = best_trocr_text

        logger.info(
            "Line %d used TrOCR fallback via variant '%s' with score %.3f",
            idx,
            best_variant_name,
            best_score,
        )
        return line

    def _correct_lines(self, lines: List[OCRLine]) -> str:
        outputs: List[str] = []
        for line in lines:
            base_text = normalize_spaces(line.merged_text or line.raw_text)
            if not self.corrector:
                line.corrected_text = base_text
            else:
                if (
                    self.config.routing.skip_correction_for_very_short_text
                    and len(base_text) < self.config.routing.min_text_len_for_correction
                ):
                    line.corrected_text = base_text
                else:
                    line.corrected_text = self.corrector.correct(base_text)
            outputs.append(line.corrected_text)
        return normalize_spaces(" ".join(outputs))

    def process_image(self, image_path: str) -> OCRResult:
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image not found: {image_path}")

        img_bgr = cv2.imread(image_path)
        if img_bgr is None:
            raise ValueError(f"Could not read image: {image_path}")

        stage_times: Dict[str, float] = {}
        t_global = time.time()

        t0 = time.time()
        triage = ImageTriage.analyze(img_bgr)
        stage_times["triage_seconds"] = round(time.time() - t0, 4)

        t0 = time.time()
        preprocessed = self.preprocessor.process(img_bgr, triage)
        self._save_debug_image("01_preprocessed.png", preprocessed)
        stage_times["preprocess_seconds"] = round(time.time() - t0, 4)

        t0 = time.time()
        lines = self.paddle.run(preprocessed)
        stage_times["paddle_seconds"] = round(time.time() - t0, 4)

        fallback_jobs: List[Tuple[int, OCRLine]] = []
        for i, line in enumerate(lines, start=1):
            line.suspicious = self.router.is_suspicious(line)
            line.merged_text = line.raw_text

            x0, y0, x1, y1 = line.bbox
            if (x1 - x0) < self.config.routing.min_line_width or (y1 - y0) < self.config.routing.min_line_height:
                continue

            if self.trocr and self.router.needs_fallback(line):
                fallback_jobs.append((i, line))

        t0 = time.time()
        if self.trocr and fallback_jobs:
            with ThreadPoolExecutor(max_workers=self.config.routing.max_fallback_workers) as executor:
                future_map = {
                    executor.submit(self._process_fallback_line, preprocessed, line, idx): (idx, line)
                    for idx, line in fallback_jobs
                }
                for future in as_completed(future_map):
                    idx, original_line = future_map[future]
                    try:
                        processed_line = future.result()
                        original_line.merged_text = processed_line.merged_text
                        original_line.fallback_used = processed_line.fallback_used
                        original_line.chosen_variant = processed_line.chosen_variant
                        original_line.candidate_scores = processed_line.candidate_scores
                    except Exception as exc:
                        logger.warning("Parallel fallback failed on line %d: %s", idx, exc)
        stage_times["fallback_seconds"] = round(time.time() - t0, 4)

        t0 = time.time()
        raw_text = normalize_spaces(
            " ".join(
                (ln.merged_text or ln.raw_text)
                for ln in lines
                if (ln.merged_text or ln.raw_text).strip()
            )
        )
        corrected_text = self._correct_lines(lines)
        stage_times["correction_seconds"] = round(time.time() - t0, 4)

        runtime = time.time() - t_global
        metadata = {
            "runtime_seconds": round(runtime, 3),
            "num_lines": len(lines),
            "fallback_lines": sum(1 for ln in lines if ln.fallback_used),
            "suspicious_lines": sum(1 for ln in lines if ln.suspicious),
            "quality_mode": self.config.routing.quality_mode,
            "cache_entries": len(self.crop_cache),
            **stage_times,
        }

        return OCRResult(
            image_path=image_path,
            raw_text=raw_text,
            corrected_text=corrected_text,
            lines=lines,
            triage=triage,
            metadata=metadata,
        )

    def export_result_json(self, result: OCRResult, output_path: str) -> None:
        payload = {
            "image_path": result.image_path,
            "raw_text": result.raw_text,
            "corrected_text": result.corrected_text,
            "triage": result.triage.__dict__,
            "metadata": result.metadata,
            "lines": [
                {
                    "bbox": line.bbox,
                    "raw_text": line.raw_text,
                    "confidence": line.confidence,
                    "source": line.source,
                    "merged_text": line.merged_text,
                    "corrected_text": line.corrected_text,
                    "fallback_used": line.fallback_used,
                    "suspicious": line.suspicious,
                    "chosen_variant": line.chosen_variant,
                    "candidate_scores": line.candidate_scores,
                    "edit_ops": levenshtein_ops(line.merged_text or line.raw_text, line.corrected_text or ""),
                }
                for line in result.lines
            ],
        }
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)


def _run_self_tests() -> None:
    assert normalize_spaces("a   b\n c") == "a b c"
    assert weird_char_ratio("") == 1.0
    assert digit_ratio("abc123") == 0.5

    ops = levenshtein_ops("cat", "coat")
    assert isinstance(ops, list)
    assert len(ops) >= 1

    score_clean = FusionEngine.language_plausibility_score("this is a clean sentence")
    score_noisy = FusionEngine.language_plausibility_score("%%% 9999 ???")
    assert score_clean > score_noisy

    print("Self-tests passed.")
    if PaddleOCR is None:
        print("Note: PaddleOCR is not installed; OCR runtime is unavailable until you install it.")


In [4]:
image_path = r"C:\Users\abuba\OneDrive\Desktop\New folder\test_image.png"

config = AppConfig(save_debug_images=True)
config.routing.quality_mode = "best"

app = DyslexiaOCRApp(config)
result = app.process_image(image_path)
app.export_result_json(result, "ocr_result.json")

print("\n===== RAW OCR =====\n")
print(result.raw_text)
print("\n===== CORRECTED OCR =====\n")
print(result.corrected_text)
print("\n===== METADATA =====\n")
print(json.dumps(result.metadata, indent=2))

2026-03-03 15:08:06,536 | INFO | Loading PaddleOCR on CPU...
C:\Users\abuba\AppData\Local\Temp\ipykernel_14372\2197695857.py:380: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  self.ocr = PaddleOCR(


ValueError: Unknown argument: use_gpu